# Validación: embeddings de reseñas + UMAP (2D / 3D)

**Objetivo:** comprobar en local que el flujo *texto → embeddings → UMAP → scatter* corre sin errores, en tiempo razonable sobre una **muestra** de reseñas, y que el mapa se puede colorear con una etiqueta conocida (p. ej. sentimiento) para interpretación exploratoria.

Esto no sustituye métricas del modelo de sentimiento; sirve como exploración del corpus antes de integrar nada en el pipeline o el dashboard.

**Instalación:** celda siguiente — opción A `pip install -r ABSA/requirements.txt` desde `notebooks/`, u opción B paquetes mínimos listados allí. Para leer etiquetas desde **Neon**, además `sqlalchemy` y `psycopg2-binary`, y en la configuración `DATA_SOURCE = "neon"` con `DATABASE_URL` en el entorno.

**Embeddings:** en la celda de configuración, `EMBEDDING_BACKEND = "robertuito"` usa el mismo modelo que el pipeline (`pysentimiento/robertuito-sentiment-analysis`): vector = *mean pooling* de la última capa oculta (no la probabilidad de clase), más alineado a polaridad que MiniLM genérico.

## Dependencias

Opción A (mismo stack que ABSA/BERTopic en el repo):

```bash
pip install -r ABSA/requirements.txt
```

Opción B (mínimo para este notebook):

```bash
pip install sentence-transformers umap-learn pandas numpy matplotlib tqdm plotly "nbformat>=4.2.0" ipykernel
```

Si usás **`EMBEDDING_BACKEND = "robertuito"`**, además: `pip install transformers torch` (y opcional `accelerate`). Robertuito trunca a **128 tokens** por reseña.

Si cargás datos desde **Neon** (`DATA_SOURCE = "neon"` en la celda de configuración), añadí conexión a Postgres:

```bash
pip install sqlalchemy psycopg2-binary
```

Opcional: `pip install python-dotenv` para cargar `notebooks/.env` automáticamente (la celda de config intenta `load_dotenv()` si está instalado).

**Gráfico 3D interactivo (rotar con el mouse):**

- Recomendado en Cursor/VS Code: `pip install plotly "nbformat>=4.2.0" ipykernel` y en la config `INTERACTIVE_3D = "plotly"` (sin `nbformat`, `fig.show()` puede fallar).
- Alternativa Jupyter clásica: `pip install ipympl` y `INTERACTIVE_3D = "ipympl"` (usa `%matplotlib widget`).
- Sin paquetes extra: `INTERACTIVE_3D = "static"` (figura estática `inline`).

La primera ejecución descarga el modelo de Hugging Face (`~/.cache/huggingface`). Con GPU, `sentence-transformers` la usará si PyTorch ve CUDA; en CPU el tiempo crece: reduce `MAX_DOCS` o el `batch_size`.

**Neon:** en el panel de Neon copiá la *connection string* y definila en el entorno (no la pegues en el notebook si el repo es público):

- PowerShell: `$env:DATABASE_URL = "postgresql://..."`
- Bash: `export DATABASE_URL="postgresql://..."`

También se acepta la variable `NEON_DATABASE_URL` si preferís separarla del resto del proyecto.

In [1]:
import os
from pathlib import Path

# --- Origen: "csv" (archivo local) o "neon" (PostgreSQL / Neon) ---
DATA_SOURCE = "neon"  # cambia a "neon" para leer reviews + sentimiento desde la BD

# URL de conexión (Neon la muestra como "connection string"). No la guardes en el repo.
try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass
DATABASE_URL = os.environ.get("DATABASE_URL") or os.environ.get("NEON_DATABASE_URL") or ""

# Máximo de filas traídas de Neon antes del muestreo local (evita traer toda la tabla)
NEON_SQL_LIMIT = 10_000

# --- Reproducibilidad y tamaño de muestra ---
RANDOM_STATE = 42
MAX_DOCS = 1500  # sube con cuidado en CPU

# --- Solo si DATA_SOURCE == "csv" ---
CSV_PATH = Path("../sources/data/Original/resenas_hotel.csv")
CSV_SEP = ","
CSV_ENCODING = "utf-8"

# Columnas en el CSV (si usás neon, la celda de carga fija TEXT_COL / COLOR_COL al esquema del repo)
TEXT_COL = "Reseña_Completa"
COLOR_COL = "Rating"  # o None; en Neon será predicciones_sentimiento.sentimiento

# --- Embeddings / cómputo ---
# "sentence_transformers": MiniLM multilingüe (similitud general)
# "robertuito": mismo checkpoint que el pipeline (español; vector = mean-pool capa oculta, alineado a sentimiento)
EMBEDDING_BACKEND = "robertuito"  # "sentence_transformers" | "robertuito"
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"  # solo si sentence_transformers
ROBERTUITO_MODEL_ID = "pysentimiento/robertuito-sentiment-analysis"
ROBERTUITO_MAX_LENGTH = 128
ENCODE_BATCH_SIZE = 32

# --- UMAP (hiperparámetros exploratorios; cambian la forma del mapa) ---
UMAP_N_COMPONENTS = 3  # 2 = solo mapa 2D; 3 = permite scatter 3D (más lento que 2D)
UMAP_N_NEIGHBORS = 8
UMAP_MIN_DIST = 0.3
UMAP_METRIC = "cosine"

# Gráfico: True = figura 3D; False = solo 2D (primeras 2 coords de UMAP)
PLOT_3D = True

# 3D interactivo: "plotly" (recomendado), "ipympl" (%matplotlib widget), "static" (sin rotar)
INTERACTIVE_3D = "plotly"

In [2]:
import pandas as pd
import numpy as np

# Esquema alineado con guidelines/ddl_bd.sql: reviews.texto_limpio + predicciones_sentimiento.sentimiento
NEON_QUERY = """
SELECT
  r.id AS review_id,
  r.texto_limpio,
  p.sentimiento
FROM reviews r
INNER JOIN predicciones_sentimiento p ON p.review_id = r.id
ORDER BY r.id DESC
LIMIT :lim
"""

if DATA_SOURCE == "neon":
    if not DATABASE_URL:
        raise ValueError(
            "DATA_SOURCE='neon' requiere DATABASE_URL o NEON_DATABASE_URL en el entorno. "
            "En Neon: copiar connection string y exportarla antes de ejecutar el kernel."
        )
    from sqlalchemy import create_engine, text

    engine = create_engine(DATABASE_URL, pool_pre_ping=True)
    df = pd.read_sql(text(NEON_QUERY), engine, params={"lim": int(NEON_SQL_LIMIT)})
    engine.dispose()
    TEXT_COL = "texto_limpio"
    COLOR_COL = "sentimiento"
    print(f"[neon] Filas: {len(df)}, columnas: {list(df.columns)}")
elif DATA_SOURCE == "csv":
    if not CSV_PATH.is_file():
        raise FileNotFoundError(
            f"No se encontró el CSV: {CSV_PATH.resolve()}. "
            "Ajustá CSV_PATH o usá DATA_SOURCE='neon' con DATABASE_URL."
        )
    df = pd.read_csv(CSV_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
    if TEXT_COL not in df.columns:
        raise ValueError(
            f"El CSV debe incluir la columna '{TEXT_COL}'. Columnas: {list(df.columns)}"
        )
    print(f"[csv] Filas leídas: {len(df)}, columnas: {list(df.columns)}")
else:
    raise ValueError("DATA_SOURCE debe ser 'csv' o 'neon'")

[neon] Filas: 5445, columnas: ['review_id', 'texto_limpio', 'sentimiento']


In [3]:
_color = COLOR_COL if (COLOR_COL and COLOR_COL in df.columns) else None
df_work = df[[TEXT_COL] + ([_color] if _color else [])].copy()
df_work = df_work.dropna(subset=[TEXT_COL])
df_work[TEXT_COL] = df_work[TEXT_COL].astype(str).str.strip()
df_work = df_work[df_work[TEXT_COL].str.len() > 0]

n = min(MAX_DOCS, len(df_work))
df_sample = df_work.sample(n=n, random_state=RANDOM_STATE).reset_index(drop=True)
textos = df_sample[TEXT_COL].tolist()
print(f"Muestra para embedding/UMAP: {len(textos)} documentos")

Muestra para embedding/UMAP: 1500 documentos


In [4]:
import numpy as np
import torch
from tqdm.auto import tqdm

if EMBEDDING_BACKEND == "sentence_transformers":
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(
        textos,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
elif EMBEDDING_BACKEND == "robertuito":
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(ROBERTUITO_MODEL_ID)
    model = AutoModel.from_pretrained(ROBERTUITO_MODEL_ID)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    def _mean_pool(last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = (last_hidden * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        return summed / denom

    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(textos), ENCODE_BATCH_SIZE)):
            batch = textos[start : start + ENCODE_BATCH_SIZE]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=ROBERTUITO_MAX_LENGTH,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
            chunks.append(pooled.cpu().numpy())
    embeddings = np.vstack(chunks)
else:
    raise ValueError("EMBEDDING_BACKEND debe ser 'sentence_transformers' o 'robertuito'")

print("Backend:", EMBEDDING_BACKEND, "| Shape embeddings:", embeddings.shape)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/47 [00:00<?, ?it/s]

Backend: robertuito | Shape embeddings: (1500, 768)


In [5]:
from umap import UMAP

reducer = UMAP(
    n_components=UMAP_N_COMPONENTS,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE,
)
xy = reducer.fit_transform(embeddings)
print("Shape UMAP:", xy.shape)

c:\Estudio\Maestria\Tesis\notebooks\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Estudio\Maestria\Tesis\notebooks\.venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
c:\Estudio\Maestria\Tesis\notebooks\.venv\Lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Shape UMAP: (1500, 3)


In [6]:
try:
    from IPython import get_ipython
except ImportError:

    def get_ipython():
        return None


_ip = get_ipython()
PLOT_3D_ACTIVE = PLOT_3D and xy.shape[1] >= 3
INTERACTIVE_MODE = str(globals().get("INTERACTIVE_3D", "plotly")).lower()

# ipympl: activar widget ANTES de importar pyplot (modo matplotlib interactivo)
if PLOT_3D_ACTIVE and INTERACTIVE_MODE == "ipympl" and _ip is not None:
    try:
        import ipympl  # noqa: F401

        _ip.run_line_magic("matplotlib", "widget")
    except Exception as exc:
        print(
            "No se pudo activar ipympl (%r). Probá: pip install ipympl o INTERACTIVE_3D='plotly'."
            % (exc,)
        )

import matplotlib.pyplot as plt
from matplotlib import colormaps

_hue = COLOR_COL if (COLOR_COL and COLOR_COL in df_sample.columns) else None
labels = cats = cmap = None
if _hue:
    labels = df_sample[_hue].astype(str)
    cats = sorted(labels.unique(), key=lambda x: (len(x), x))
    cmap = colormaps["tab10"].resampled(max(len(cats), 1))

kw2 = {"s": 14, "alpha": 0.55}
kw3 = {"s": 14, "alpha": 0.55, "depthshade": True}

drew_plotly_3d = False
if PLOT_3D_ACTIVE and INTERACTIVE_MODE == "plotly":
    try:
        import plotly.express as px

        _plot_df = pd.DataFrame(
            {"UMAP-1": xy[:, 0], "UMAP-2": xy[:, 1], "UMAP-3": xy[:, 2]}
        )
        if _hue:
            _plot_df[_hue] = labels.values
        fig_p = px.scatter_3d(
            _plot_df,
            x="UMAP-1",
            y="UMAP-2",
            z="UMAP-3",
            color=_hue if _hue else None,
            title=(f"UMAP 3D (Plotly) — color = {_hue}" if _hue else "UMAP 3D (Plotly)"),
            opacity=0.55,
        )
        fig_p.update_traces(marker=dict(size=4))
        try:
            fig_p.show()
        except ValueError as exc:
            err = str(exc).lower()
            if "nbformat" in err or "mime type" in err:
                print(
                    "Plotly en Jupyter necesita nbformat>=4.2. Ejecutá:\n"
                    '  pip install "nbformat>=4.2.0" ipykernel\n'
                    "Luego reiniciá el kernel. Se dibuja matplotlib 3D abajo."
                )
            else:
                raise
        else:
            drew_plotly_3d = True
    except ImportError:
        print("Plotly no instalado. Ejecutá: pip install plotly")

drew_mpl = False
if PLOT_3D_ACTIVE and not drew_plotly_3d:
    fig = plt.figure(figsize=(11, 9))
    ax = fig.add_subplot(111, projection="3d")
    if _hue:
        for i, c in enumerate(cats):
            m = labels == c
            ax.scatter(xy[m, 0], xy[m, 1], xy[m, 2], label=c, color=cmap(i), **kw3)
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        ax.set_title(f"UMAP 3D — color = {_hue}")
    else:
        ax.scatter(xy[:, 0], xy[:, 1], xy[:, 2], c="steelblue", **kw3)
        ax.set_title("UMAP 3D (sin columna de color)")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_zlabel("UMAP-3")
    drew_mpl = True
elif not PLOT_3D_ACTIVE:
    fig, ax = plt.subplots(figsize=(10, 8))
    if _hue:
        for i, c in enumerate(cats):
            m = labels == c
            ax.scatter(xy[m, 0], xy[m, 1], label=c, color=cmap(i), **kw2)
        ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
        ax.set_title(f"UMAP 2D — color = {_hue}")
    else:
        ax.scatter(xy[:, 0], xy[:, 1], c="steelblue", **kw2)
        ax.set_title("UMAP 2D (sin columna de color en la muestra)")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    drew_mpl = True

if drew_mpl:
    plt.tight_layout()
    plt.show()

## Interpretación

- **UMAP es exploratorio:** la posición relativa depende de hiperparámetros (`n_neighbors`, `min_dist`) y de la muestra; no interpretes distancias absolutas como probabilidades.
- **Validación útil:** si tienes `sentimiento` o rating, revisa si regiones del mapa mezclan etiquetas (solapamiento) o si hay grupos coherentes.
- **Vista 3D interactiva:** configurá `INTERACTIVE_3D`: `"plotly"` (instalar `plotly`) o `"ipympl"` (instalar `ipympl` y, si hace falta, reiniciar el kernel tras la primera vez). Con `"static"` el 3D es solo imagen (`inline`).
- **Siguiente paso cualitativo:** elige 3–5 puntos en una región densa y lee el texto original en `df_sample` usando los índices del scatter (por ejemplo los más cercanos al centroide de un cluster manual o los vecinos más cercanos en el espacio de embeddings).

Para integración futura con el dashboard, conviene **precomputar** coordenadas 2D en batch y servirlas por API, en lugar de ejecutar UMAP en cada visita al navegador.